In [ ]:
from dataclasses import dataclass
from pathlib import Path
import matplotlib.pyplot as plt
import prism
import warnings
import numpy as np
import time  
  

from imagematerials.sensitivity_analysis.changedata import change_sector, ChangeAction, ChangeReplace
from imagematerials.factory import ModelFactory
from imagematerials.maintenance import Maintenance
from imagematerials.model import GenericMaterials, GenericStocks
from imagematerials.preprocessing import get_preprocessing_data

from imagematerials.sensitivity_analysis.monte_carlo import load_ranges_material_intensities, sample_material_intensities, load_ranges_lifetimes

warnings.filterwarnings("ignore")

path_current = Path().resolve()
path_base = path_current.parent #.parent # base path of the project -> image-materials
path_base = Path(path_base, "data", "raw")

In [ ]:
# Get the preprocessing data for the vehicles sector only once
vhc_sector = get_preprocessing_data(
    "vehicles", Path("..", "data", "raw"), 
    climate_policy_scenario_dir = Path("..", "data", "raw", "image", "SSP2_baseline"), 
    circular_economy_scenario_dirs = None
)

## test MC functions

In [ ]:
ranges_lt = load_ranges_lifetimes(path_base / "vehicles" / "test_lifetimes_years.csv")
rng = np.random.default_rng(42)   # seed once for reproducibility
lt = sample_intensities("vehicles",ranges_lt, rng=rng)

In [ ]:
ranges = load_ranges_materials(path_base / "vehicles" / "test_all_vehicles_material_ranges.csv")
rng = np.random.default_rng(42)   # seed once for reproducibility
mi = sample_intensities("vehicles",ranges, rng=rng)

In [ ]:
ranges

## test model run

In [ ]:
time_start = 2000
time_end = 2055
complete_timeline = prism.Timeline(time_start, time_end, 1)
simulation_timeline = prism.Timeline(time_start, time_end, 1)

ranges = load_ranges_materials(path_base / "vehicles" / "test_all_vehicles_material_ranges.csv")
ranges = ranges.loc[ranges["Cohort"]==2020]
rng = np.random.default_rng(42)   # seed once for reproducibility

start = time.time()
all_output = {}
N = 15
for i in range(N):
    mi = sample_intensities("vehicles", ranges, rng=rng, year_start=1807, year_end=2100)   # same shape as original
    change_definition = {
        "material_fractions": ChangeReplace(mi)
    }
    new_vhc_sector = change_sector(vhc_sector, change_definition, inplace=False)
    factory = ModelFactory(
        new_vhc_sector, complete_timeline
        ).add(GenericStocks
        ).add(GenericMaterials
        ).add(Maintenance
        )
    model = factory.finish()
    model.simulate(simulation_timeline)
    all_output[i] = model
    print(f"\rSimulation {i} completed.     ", end="")

end = time.time()
print(f"Total time for {N} simulations: {(end - start)/60:.1f} minutes.")

In [ ]:
fig, ax = plt.subplots()
for i in range(N):
    mf = all_output[i].vehicles["material_fractions"]
    plt.plot(mf.Cohort, mf.sel(Type='Cars - ICE', material='aluminium'), label=f"Simulation {i+1}")
plt.xlabel("Time")
plt.ylabel("Material Fractions")
plt.title("Varying Model Input")
# plt.legend()
plt.show()

In [ ]:
fig, ax = plt.subplots()
for i in range(N):
    mf = all_output[i].vehicles["inflow_materials"].to_array().sel(time=slice(2005, None)).sum("Region")
    plt.plot(mf.time, mf.sel(Type='Cars - ICE', material='aluminium'), label=f"Simulation {i+1}")
plt.xlabel("Time")
plt.ylabel("Inflow Materials")
plt.title("Varying material demand depending on material intensity")
# plt.legend()
plt.show()

## test other change actions

In [ ]:
@dataclass
class ChangeFirstElementIn3DArray(ChangeAction):
    new_value: float

    def apply(self, value):
        value[0, 0, 0] = self.new_value
        return value

In [ ]:
list_of_values = [0.42, 0.41, 0.40, 0.39, 0.38]
for value in list_of_values:
    change_definition = {
        "material_fractions": ChangeFirstElementIn3DArray(value)
    }
    new_vhc_sector = change_sector(vhc_sector, change_definition, inplace=False)
    print(
        f"Old value: {float(vhc_sector.prep_data['material_fractions'][0, 0, 0])};"
        f" new value: {float(new_vhc_sector.prep_data['material_fractions'][0, 0, 0])}."
    )
    # ... and then run the model.